# Databricks volumes mimic

In [1]:
!tree /Volumes/workspace
!tree /Volumes/workspace/default/on_premises_certificates/postgres

/Volumes/workspace
└── default
    ├── on_premises_artifacts
    │   └── python-wheels
    │       ├── squid_databricks-0.1.0-py312-none-any.whl
    │       ├── squid_on_premises-0.1.0-py311-none-any.whl
    │       ├── squid_on_premises-0.1.0-py312-none-any.whl
    │       └── squid_on_premises-0.1.0-py313-none-any.whl
    └── on_premises_certificates
        ├── kafka-4
        │   └── ca.crt
        ├── postgres
        │   ├── ca.crt
        │   ├── client.crt
        │   ├── client.key
        │   ├── postgres-client.p12
        │   ├── secret-client-password
        │   └── secret-postgres-password
        ├── seaweedfs
        │   └── ca.crt
        └── trino
            ├── ca.crt
            ├── secret-client-password
            └── trino-client.p12

9 directories, 15 files
/Volumes/workspace/default/on_premises_certificates/postgres
├── ca.crt
├── client.crt
├── client.key
├── postgres-client.p12
├── secret-client-password
└── secret-postgres-password

1 directory, 6 files


In [5]:
import sys
from pathlib import Path

src_path = Path("~/work/on-premises/squid-test/src").expanduser().resolve()

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

In [6]:
!nc -vz postgres 5432

Connection to postgres (172.23.0.3) 5432 port [tcp/postgresql] succeeded!


In [7]:
from squid.resources.postgres import get_cursor

ModuleNotFoundError: No module named 'squid'

In [5]:
%pip install psycopg2-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 34.9 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [6]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

src_path = Path("~/work/on-premises/squid-test/src").expanduser().resolve()

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))


In [7]:
from squid.resources.postgres import get_cursor
cursor = get_cursor()

if cursor is not None:
    
    cursor.execute("""
        SELECT
            current_user AS database_user,
            ssl,
            version AS tls_version,
            cipher AS tls_cipher,
            bits AS tls_bits,
            client_dn
        FROM pg_stat_ssl
        WHERE pid = pg_backend_pid()
    """)
    
    results = cursor.fetchall()
    columns = [desc[0] for desc in cursor.description]
else:
    print("Error")

In [16]:
import pandas as pd
test_df = pd.DataFrame(results, columns=columns)
display(test_df)

,database_user,ssl,tls_version,tls_cipher,tls_bits,client_dn
0,postgres,True,TLSv1.3,TLS_AES_256_GCM_SHA384,256,/CN=postgres


In [15]:
!ls /Volumes/workspace/default/on_premises_certificates/postgres

ca.crt	    client.key		 secret-client-password
client.crt  postgres-client.p12  secret-postgres-password


In [13]:
import squid.core.main
squid.core.main.env

<Environment.ON_PREMISES: 4>